# Example Literature Project

## Helper Functions

In [1]:
import json
import textwrap
from pathlib import Path
from time import perf_counter

from kilin.conversations import ConversationStore
from kilin.rag import RAGSession


def _print_wrapped(label, value, width=79):
    """Print text with a fixed display width, preserving paragraphs."""
    print(f"{label}:")
    for line in str(value).splitlines() or [""]:
        if not line.strip():
            print()
        else:
            print(
                textwrap.fill(
                    line,
                    width=width,
                    break_long_words=False,
                    break_on_hyphens=False,
                )
            )
    print("")


def create_conversation(project_dir):
    """Create an independent conversation and its active RAG session.

    Parameters
    ----------
    project_dir : str or Path
        Existing KíLín project directory.

    Returns
    -------
    tuple
        Conversation store, conversation record, and active session.
        Close the session after the last question.

    """
    store = ConversationStore(Path(project_dir))
    conversation = store.create()
    path = store.state_path(conversation)
    session = RAGSession(
        Path(project_dir),
        conversation_path=path,
        router_model="Qwen/Qwen3-4B-MLX-4bit",
    )
    print(f"Conversation: {path}")
    return store, conversation, session





/Users/giovannivolpe/Documents/GitHub/KiLin/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### GUI-equivalent decision callbacks

In [2]:
def _show_gui_choice(question, options, selected):
    """Print the offered options and the scripted GUI selection.

    Parameters
    ----------
    question : str
        Tool question shown by the desktop GUI.
    options : tuple of str
        Choices offered by the tool.
    selected : int or None
        Chosen zero-based index, or None for cancellation.

    """
    print(f"GUI choice requested: {question}")
    for index, option in enumerate(options):
        marker = "*" if index == selected else " "
        print(f"  {marker} {index}: {option}")
    if selected is None:
        print("  Decision: cancel (no scripted choice matched)")
    else:
        print(f"  Decision: option {selected}")


def choose_first_20(question, options):
    """Choose the GUI's first-20-items chat display, when offered.

    Parameters
    ----------
    question : str
        Tool choice prompt.
    options : tuple of str
        Offered choices.

    Returns
    -------
    int or None
        Matching option index, or None when not offered.

    """
    selected = next(
        (i for i, value in enumerate(options)
         if value == "Show the first 20 here"),
        None,
    )
    _show_gui_choice(question, options, selected)
    return selected


def choose_save_all(question, options):
    """Choose the GUI option to save all matches as Markdown.

    Parameters
    ----------
    question : str
        Tool choice prompt.
    options : tuple of str
        Offered choices.

    Returns
    -------
    int or None
        Export option index, or None when not offered.

    """
    selected = next(
        (
            i for i, value in enumerate(options)
            if value.startswith("Save all ")
            and value.endswith(" to outputs/ as Markdown")
        ),
        None,
    )
    _show_gui_choice(question, options, selected)
    return selected


def choose_gui_default(question, options):
    """Use a documented automatic selection for common GUI choices.

    Long lists use the GUI's 20-item display. An ambiguous document
    question chooses the first offered paper, as if the user clicked the
    first GUI option. Every decision is printed; other choices cancel.

    Parameters
    ----------
    question : str
        Tool question.
    options : tuple of str
        Offered choices.

    Returns
    -------
    int or None
        Selected option index, or None to cancel.

    """
    selected = next(
        (i for i, value in enumerate(options)
         if value == "Show the first 20 here"),
        None,
    )
    if selected is None and options and (
        "which paper" in question.casefold()
        or "which document" in question.casefold()
    ):
        selected = 0
    _show_gui_choice(question, options, selected)
    return selected


def choose_document_named(filename):
    """Create a callback that selects one named file in a GUI prompt.

    Parameters
    ----------
    filename : str
        Filename to select, including its extension.

    Returns
    -------
    callable
        Callback with the GUI's ``(question, options)`` signature.

    """
    def choose(question, options):
        """Select the requested file from the currently offered options."""
        selected = next(
            (
                index for index, value in enumerate(options)
                if value.split(" (", 1)[0].casefold()
                == filename.casefold()
            ),
            None,
        )
        _show_gui_choice(question, options, selected)
        return selected

    return choose


def _confirm_gui_plan(plan, selection):
    """Report a GUI-equivalent plan decision, disallowing web access.

    Parameters
    ----------
    plan : ToolPlan
        Plan offered by the tool.
    selection : str
        Intended choice: continue, quick, or cancel.

    Returns
    -------
    str
        Applied choice, with network requests always cancelled.

    """
    chosen = (
        "cancel" if getattr(plan, "operation", "model") == "network"
        else selection
    )
    print(f"GUI plan: {plan.tool} | {plan.summary}")
    print(
        f"  Items: {plan.items}, batches: {plan.batches_to_run}, "
        f"quick items: {plan.quick_items}"
    )
    print(f"  Decision: {chosen}")
    return chosen


def confirm_gui_continue(plan):
    """Simulate clicking Continue, but never approve network lookup."""
    return _confirm_gui_plan(plan, "continue")


def confirm_gui_quick(plan):
    """Simulate clicking Quick, but never approve network lookup."""
    return _confirm_gui_plan(plan, "quick")


def confirm_gui_cancel(plan):
    """Simulate clicking Cancel on a tool's proposed plan."""
    return _confirm_gui_plan(plan, "cancel")



### Question and trace inspection

In [3]:
def ask_question(
    session,
    question,
    *,
    choice_callback=choose_gui_default,
    confirm_callback=confirm_gui_continue,
):
    """Ask the active model and print a width-limited exchange.

    Parameters
    ----------
    session : RAGSession
        Active conversation session.
    question : str
        User question.
    choice_callback : callable or None, optional
        Select a GUI tool option. The default displays the first 20
        matches or selects the first offered paper. None deliberately
        uses the noninteractive fallback.
    confirm_callback : callable or None, optional
        Select Continue, Quick, or Cancel for an expensive tool plan.
        The default selects Continue; network plans always cancel.

    Returns
    -------
    str
        Assistant response.

    """
    print("\n" + "=" * 79)
    _print_wrapped("USER", question)
    start = perf_counter()
    try:
        answer = session.ask(
            question,
            retrieval_count=10,
            choice_callback=choice_callback,
            confirm_callback=confirm_callback,
        )
    except Exception:
        print(f"Elapsed before failure: {perf_counter() - start:.3f} s")
        raise
    _print_wrapped("ASSISTANT", answer)
    if session.turns:
        parameters = session.turns[-1].parameters
        tool = parameters.get("tool", {})
        print(f"Resolved intent: {parameters.get('intent', 'unknown')}")
        if isinstance(tool, dict) and tool.get("name"):
            print(f"Executed tool: {tool['name']}")
    print(f"Wall time: {perf_counter() - start:.3f} s")
    return answer


def _print_retrieved_chunks(stages, width=79):
    """Display retrieved passages recorded in the native trace.

    Parameters
    ----------
    stages : dict
        Stages of one recorded conversation turn.
    width : int, optional
        Maximum display width.

    """
    retrieval = stages.get("03_retrieval", {})
    operations = retrieval.get("operations", {})

    print("\nRETRIEVED CHUNKS")

    found = 0

    def visit(value, path):
        nonlocal found

        if isinstance(value, dict):
            text = next(
                (
                    value[key]
                    for key in (
                        "text",
                        "content",
                        "chunk_text",
                        "passage",
                        "snippet",
                    )
                    if isinstance(value.get(key), str) and value[key].strip()
                ),
                None,
            )

            if text is not None:
                found += 1
                print(f"\n  CHUNK {found}")
                print(f"  Trace location: {path}")

                for key, item in value.items():
                    if key in (
                        "text",
                        "content",
                        "chunk_text",
                        "passage",
                        "snippet",
                    ):
                        continue

                    if isinstance(item, (str, int, float, bool)):
                        _print_wrapped(f"  {key}", item, width)

                _print_wrapped("  PASSAGE", text, width)
                return

            for key, child in value.items():
                visit(child, f"{path}/{key}")

        elif isinstance(value, list):
            for index, child in enumerate(value):
                visit(child, f"{path}/{index}")

    for name, operation in operations.items():
        if "retrieval" not in name.lower():
            continue

        visit(operation.get("details", {}), name)

    if not found:
        print("  No passage text found in the retrieval trace.")


def inspect_conversation(
    conversation_path,
    turn_number=None,
    show_chunks=True,
):
    """Display routing, retrieval, model calls, tools, and trace timings.

    Parameters
    ----------
    conversation_path : str or Path
        Native KíLín trace JSON file.
    turn_number : int, optional
        One-based turn index, or None for all turns.
    show_chunks : bool, optional
        Whether to display passages recorded in the retrieval trace.
        Defaults to True.

    Returns
    -------
    dict
        Unmodified native trace record.

    """
    with Path(conversation_path).open(encoding="utf-8") as file:
        trace = json.load(file)

    turns = list(trace["sessions"]["05_conversations"]["turns"].items())
    if turn_number is not None:
        if not 1 <= turn_number <= len(turns):
            raise ValueError("Invalid turn number.")
        turns = [turns[turn_number - 1]]

    def operation(stages, stage_name, operation_name):
        """Find a stage operation even if its sequence number changed."""
        operations = stages.get(stage_name, {}).get("operations", {})
        exact = operations.get(operation_name)
        if exact is not None:
            return exact
        suffix = "_" + operation_name.split("_", 1)[1]
        return next(
            (entry for key, entry in operations.items()
             if key.endswith(suffix)),
            {},
        )

    def show_fields(heading, data, keys):
        print(f"\n{heading}")
        for key in keys:
            if key in data:
                _print_wrapped(f"  {key}", data[key])

    for turn_id, turn in turns:
        stages = turn.get("stages", {})
        print("\n" + "=" * 79)
        print(f"TURN: {turn_id}")
        print("=" * 79)

        question = operation(stages, "01_question", "01_turn")
        response = operation(
            stages, "01_question", "09_conversation_turn_representation"
        )
        show_fields("EXCHANGE", question.get("details", {}), ["question"])
        show_fields("ANSWER", response.get("details", {}), ["summary"])

        decision = operation(
            stages, "01_question", "05_intent_classification_decided"
        ).get("details", {})
        show_fields(
            "INTENT",
            decision.get("classification", {}),
            ["intent", "standalone", "refers_back", "related_turn_ids"],
        )
        show_fields("ROUTING", decision, ["shortcut", "fallback"])
        show_fields(
            "RESOLVED POLICY",
            decision.get("resolved", {}),
            ["history_policy", "retrieval_mode", "retrieval_strategy"],
        )
        settings = operation(
            stages, "01_question", "02_turn_settings_snapshot"
        ).get("details", {})
        show_fields(
            "EFFECTIVE SETTINGS",
            settings,
            [
                "project_type_id",
                "backend",
                "chat_model",
                "router_model",
                "thinking",
                "temperature",
                "max_new_tokens",
                "retrieval_count",
            ],
        )
        retrieval = operation(stages, "03_retrieval", "02_retrieval_result")
        show_fields(
            "RETRIEVAL",
            retrieval.get("details", {}),
            ["mode", "strategy", "evidence_count", "duration_ms"],
        )
        if show_chunks:
            _print_retrieved_chunks(stages)

        calls = []
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                if name == "01_llm_call_input" or name == "02_llm_call_output":
                    calls.append((stage_name, name, entry))
                elif "tool" in name.lower():
                    print(f"\nTOOL OPERATION: {stage_name}/{name}")
                    show_fields(
                        "  DETAILS",
                        entry.get("details", {}),
                        ["tool", "tool_name", "status", "duration_ms"],
                    )
        for stage_name, name, entry in calls:
            details = entry.get("details", {})
            show_fields(
                f"MODEL CALL: {stage_name}/{name}",
                details,
                [
                    "call_id",
                    "model_id",
                    "backend",
                    "device",
                    "thinking",
                    "input_token_count",
                    "prompt_token_count",
                    "output_token_count",
                    "finish_reason",
                    "reply_duration_ms",
                ],
            )
            show_fields(
                "GENERATION SETTINGS",
                details.get("generation", {}),
                ["temperature", "max_new_tokens", "thinking"],
            )

        print("\nOPERATION TIMINGS (ms)")
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                timing = entry.get("timing", {})
                duration = timing.get("operation_duration_ms")
                if duration is not None:
                    print(f"  {stage_name}/{name}: {duration:.3f}")

    return trace




### Intent and tool verification helpers

In [4]:
# Intent coverage is based on saved turn metadata, not on the wording of
# the test question or the route we expected the model to choose.
intent_checks = []


def _intent_definitions():
    """Read the active repository's literature intent definitions."""
    path = repo_root / "kilin" / "project_types" / "literature.json"
    with path.open(encoding="utf-8") as file:
        return json.load(file)["intents"]


# Fields demonstrating that a tool progressed beyond initial selection.
# They are deliberately not assertions about scientific correctness.
_TOOL_RESULT_FIELDS = {
    "project_inventory": ("files",),
    "scan_and_list": ("total_mentions", "listed"),
    "show_source": ("turn", "passages"),
    "filter_results": ("turn", "items"),
    "paper_overview": ("paper",),
    "suggest_reference": ("statement", "papers_checked"),
    "find_passages": ("topic", "choice"),
    "find_fact": ("choice",),
    "batch_review": ("document", "choice"),
    "batch_classify": ("categories", "choice"),
    "reference_check": ("paper",),
    "generate_paper_summary": ("template", "saved"),
}


def check_intent(
    session,
    expected_intent,
    question,
    *,
    choice_callback=choose_gui_default,
    confirm_callback=confirm_gui_quick,
):
    """Execute a test and compare saved routing to its intended path.

    Parameters
    ----------
    session : RAGSession
        Active session; use a new conversation for independent groups.
    expected_intent : str
        Intended name in ``literature.json``; never forced on the router.
    question : str
        Test request sent unchanged to KíLín.
    choice_callback : callable or None, optional
        Scripted GUI choice. The default shows lists in the chat.
    confirm_callback : callable or None, optional
        Scripted GUI confirmation. Quick is the default to limit work.

    Returns
    -------
    str or None
        Answer, or None when this test failed before an answer was saved.

    """
    definitions = _intent_definitions()
    if expected_intent not in definitions:
        raise ValueError(f"Unknown literature intent: {expected_intent}")
    expected_tool = definitions[expected_intent].get("tool", {}).get("name")
    count_before = len(session.turns)
    record = {
        "expected": expected_intent,
        "expected_tool": expected_tool,
        "question": question,
    }
    try:
        answer = ask_question(
            session,
            question,
            choice_callback=choice_callback,
            confirm_callback=confirm_callback,
        )
        if len(session.turns) != count_before + 1:
            raise RuntimeError("The question did not save exactly one turn.")
        parameters = session.turns[-1].parameters
        tool = parameters.get("tool", {})
        actual_tool = tool.get("name") if isinstance(tool, dict) else None
        record.update(
            actual=parameters.get("intent"),
            actual_tool=actual_tool,
            invalid_citations=parameters.get("invalid_citations", []),
        )
        record["status"] = (
            "MATCH"
            if record["actual"] == expected_intent
            and actual_tool == expected_tool
            else "MISMATCH"
        )
        if isinstance(tool, dict):
            record["output"] = tool.get("saved") or tool.get("report")
            required = _TOOL_RESULT_FIELDS.get(expected_tool, ())
            missing = [key for key in required if key not in tool]
            if record["status"] == "MATCH" and missing:
                record["status"] = "INCOMPLETE"
                record["missing"] = missing
            # Save and report cases should create a file if advertised.
            saved = tool.get("saved")
            if saved and not (project_dir / saved).is_file():
                record["status"] = "INCOMPLETE"
                record["missing_file"] = saved
    except Exception as error:
        answer = None
        record["status"] = "ERROR"
        record["error"] = f"{type(error).__name__}: {error}"
        print(f"INTENT TEST ERROR: {record['error']}")
    intent_checks.append(record)
    print(
        f"INTENT TEST: {expected_intent} -> {record.get('actual', '-')}; "
        f"tool {expected_tool or '-'} -> "
        f"{record.get('actual_tool') or '-'}; {record['status']}"
    )
    if record.get("missing"):
        print(f"  Missing tool result fields: {record['missing']}")
    if record.get("missing_file"):
        print(f"  Missing output file: {record['missing_file']}")
    if record.get("invalid_citations"):
        print(f"  Invalid citations: {record['invalid_citations']}")
    if record.get("output"):
        print(f"  Output file: {record['output']}")
    return answer


def skip_intent(expected_intent, reason):
    """Record a test intentionally not executed in the current run."""
    if expected_intent not in _intent_definitions():
        raise ValueError(f"Unknown literature intent: {expected_intent}")
    intent_checks.append(
        {"expected": expected_intent, "status": "SKIPPED",
         "reason": reason}
    )
    print(f"SKIPPED {expected_intent}: {reason}")


def show_intent_coverage():
    """Report attempted coverage and actual tool/intent agreement.

    Returns
    -------
    dict
        Summary counts: matched, mismatched, errors, skipped, untested.

    """
    definitions = _intent_definitions()
    counts = {key: 0 for key in (
        "matched", "mismatched", "incomplete", "errors",
        "skipped", "untested"
    )}
    print("\nLITERATURE INTENT COVERAGE")
    print(f"{'Intent':<27} {'Expected tool':<24} Status")
    print("-" * 79)
    for intent, settings in definitions.items():
        records = [
            entry for entry in intent_checks
            if entry["expected"] == intent
        ]
        active = [entry for entry in records
                  if entry["status"] != "SKIPPED"]
        if not records:
            status = "NOT RUN"
            counts["untested"] += 1
        elif not active:
            status = "SKIPPED"
            counts["skipped"] += 1
        elif all(entry["status"] == "MATCH" for entry in active):
            status = "MATCH"
            counts["matched"] += 1
        elif any(entry["status"] == "ERROR" for entry in active):
            status = "ERROR"
            counts["errors"] += 1
        elif any(entry["status"] == "INCOMPLETE" for entry in active):
            status = "INCOMPLETE"
            counts["incomplete"] += 1
        else:
            status = "MISMATCH"
            counts["mismatched"] += 1
        expected_tool = settings.get("tool", {}).get("name") or "model"
        print(f"{intent:<27} {expected_tool:<24} {status}")
    print("-" * 79)
    print(f"Routing/tool matches: {counts['matched']}/{len(definitions)}")
    print("Note: matching a route does not validate scientific correctness.")
    return counts



## Create Empty Literature Project

In [5]:
from pathlib import Path

from kilin.workspace import prepare_sources_workspace

# Locate the repository independently of the notebook working directory.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError(
            "Run this notebook from within the KíLín repository."
        )
    repo_root = repo_root.parent

project_dir = repo_root / "sandbox" / "trial_literature"
project_dir.parent.mkdir(parents=True, exist_ok=True)

# Do not overwrite a project created by an earlier run.
if project_dir.exists():
    raise FileExistsError(f"Project directory already exists: {project_dir}")

project = prepare_sources_workspace(
    [],
    "literature",
    workspace_directory=project_dir,
    settings={"allow_download": False},
)

print(f"Created literature project: {project.root}")
print(f"Source directory: {project.source}")

Created literature project: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature
Source directory: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/sources


## First Conversation without Sources

In [6]:
store, conversation, session = create_conversation(project_dir)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T16-35-13.json


In [7]:
answer = ask_question(session, "hello")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
hello

ASSISTANT:
Hello!

Resolved intent: small_talk
Wall time: 3.813 s

TURN: turn_0001

EXCHANGE
  question:
hello


ANSWER
  summary:
Hello!


INTENT
  intent:
small_talk

  standalone:
True

  refers_back:
False

  related_turn_ids:
[]


ROUTING
  shortcut:
(hi|hello|hey|hiya|hej|yo|greetings|good (morning|afternoon|evening|day))(
there| kilin)?

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.002


RETRIEVED CHUNKS
  No passage text found in the retrieval trace.

MODEL CALL: 04_llm_input/01_llm_call_input
  call_id:
llm-call_efecbe7a96bc

  model_id:
Qwen/Qwen3-8B-MLX-4bit

  backend:
mlx

  device:
metal

  thinking:
False

  input_token_co

In [8]:
answer = ask_question(session, "What can you help me with?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
What can you help me with?

ASSISTANT:
I can help you with any questions or tasks related to the project evidence
provided.

Resolved intent: small_talk
Wall time: 6.658 s

TURN: turn_0002

EXCHANGE
  question:
What can you help me with?


ANSWER
  summary:
I can help you with any questions or tasks related to the project evidence
provided.


INTENT
  intent:
small_talk

  standalone:
True

  refers_back:
False

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.002


RETRIEVED CHUNKS
  No passage text found in the retrieval trace.

MODEL CALL: 04_llm_input/01_llm_call_input
  call_id:
llm-call_6203a

In [9]:
answer = ask_question(session, "What is Brownian motion?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
What is Brownian motion?

ASSISTANT:
<think>
Okay, the user is asking about Brownian motion. Let me recall what I know.
Brownian motion is the random movement of particles in a fluid, right? I need
to explain it clearly. The user might be a student or someone with a basic
science background. They probably want a straightforward definition and maybe
some context.

First, I should mention that it's the erratic movement of microscopic particles
suspended in a liquid or gas. The key point is that it's caused by collisions
with the molecules of the fluid. I remember Einstein's work on this, so I
should include that. He explained it in 1905, which helped confirm the
existence of atoms. Also, the evidence from the project might have references
to experiments, like the one by Robert Brown. Wait, the user provided project
evidence, so I need to check if there are specific sources. The user mentioned
to use the supplied project evidence, so I should look at the sources [1] and
[2] they pr

In [10]:
answer = ask_question(session, "Which articles are available in this project?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
Which articles are available in this project?

ASSISTANT:
This project has **0 source files**, all with searchable text.

**By type**


**0 files**


Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Resolved intent: project_inventory
Executed tool: project_inventory
Wall time: 4.779 s

TURN: turn_0004

EXCHANGE
  question:
Which articles are available in this project?


ANSWER
  summary:
This project has **0 source files**, all with searchable text.


INTENT
  intent:
project_inventory

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

 

In [11]:
answer = ask_question(session, "Summarize the articles in this project.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
Summarize the articles in this project.

ASSISTANT:
This project has **0 source files**, all with searchable text.

**By type**


Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Resolved intent: project_inventory
Executed tool: project_inventory
Wall time: 4.883 s

TURN: turn_0005

EXCHANGE
  question:
Summarize the articles in this project.


ANSWER
  summary:
This project has **0 source files**, all with searchable text.


INTENT
  intent:
project_inventory

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  dura

In [12]:
answer = ask_question(
    session,
    "Find articles about neural networks in this project.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Find articles about neural networks in this project.

ASSISTANT:
Found **0 sentences** about “neural networks” in **0 of 0 files**.
No sentence in scope is about this topic.

Resolved intent: topic_listing
Executed tool: find_passages
Wall time: 6.796 s

TURN: turn_0006

EXCHANGE
  question:
Find articles about neural networks in this project.


ANSWER
  summary:
Found **0 sentences** about “neural networks” in **0 of 0 files**.


INTENT
  intent:
topic_listing

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.001


RETRIEVED CHUNKS
  No passage text found i

In [13]:
session.close()

## Add Articles to Project

In [14]:
from kilin.workspace import add_project_sources

fixtures = repo_root / "tests" / "fixtures"

articles = sorted(fixtures.glob("*.pdf"))

if not articles:
    raise FileNotFoundError(f"No PDF articles found in {fixtures}")

project = add_project_sources(
    project_dir,
    articles,
)

print(f"Project: {project.root}")
print(f"Imported articles: {len(articles)}")

for article in sorted(project.source.glob("*.pdf")):
    print(f"  {article.name}")

Project: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature
Imported articles: 5
  mijalkov17braph.pdf
  mijalkov25computational.pdf
  patil25deep.pdf
  tiunn26braph2.pdf
  yao25connecting.pdf


## Document Discovery and Retrieval

In [15]:
store, conversation, session = create_conversation(project_dir)

Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T16-36-37.json


### GUI-equivalent choices for long mention lists

The next two calls explicitly select the GUI's **Show first 20** and
**Save all** options. A third call disables the callback and exercises
the notebook-only `MAX_LISTED` fallback (100). The selected output limit
does not change the total number of matching mentions.


In [16]:
answer = ask_question(session, "Which articles are available in this project?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Which articles are available in this project?

ASSISTANT:
This project has **5 source files**, all with searchable text.

**By type**

- PDF: 5

**5 files**

- mijalkov17braph.pdf (78 parts, 15.8 MB)
- mijalkov25computational.pdf (93 parts, 6.3 MB)
- patil25deep.pdf (114 parts, 6.4 MB)
- tiunn26braph2.pdf (36 parts, 3.1 MB)
- yao25connecting.pdf (130 parts, 3.5 MB)

Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Resolved intent: project_inventory
Executed tool: project_inventory
Wall time: 5.811 s

TURN: turn_0001

EXCHANGE
  question:
Which articles are available in this project?


ANSWER
  summary:
This project has **5 source files**, all with searchable text.


INTENT
  intent:
project_inventory

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTIN

In [17]:
# GUI choice: display the first 20 mentions.
answer = ask_question(
    session,
    "Which of these articles discuss BRAPH?",
    choice_callback=choose_first_20,
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)



USER:
Which of these articles discuss BRAPH?

GUI choice requested: Found 183 mentions of “BRAPH”. Save the full list as a Markdown file?
    0: Save all 183 to outputs/ as Markdown
  * 1: Show the first 20 here
  Decision: option 1
ASSISTANT:
Found **183 mentions** of “BRAPH” in **5 of 5 files** (ignoring case, whole
words).

**Mentions per file**

- tiunn26braph2.pdf: 85
- mijalkov17braph.pdf: 80
- mijalkov25computational.pdf: 9
- patil25deep.pdf: 7
- yao25connecting.pdf: 2

**Where**

1. mijalkov17braph.pdf, page 1, line 3: **BRAPH**: A graph theory software for
the analysis of brain connecti… [1]
2. mijalkov17braph.pdf, page 1, line 27: …, for the Alzheimer’s Disease
Neuroimaging Initiative (2017) **BRAPH**: A graph theory software for the
analysis of brain [2]
3. mijalkov17braph.pdf, page 1, line 66: …ecture. We have developed a freeware
MatLab-based software (**BRAPH**–BRain Analysis using graPH the- ory) for
connectivity analy… [3]
4. mijalkov17braph.pdf, page 1, line 69: …sion

In [18]:
# GUI choice: save the complete list as markdown.
answer = ask_question(
    session,
    "Which of these articles discuss BRAPH?",
    choice_callback=choose_save_all,
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)



USER:
Which of these articles discuss BRAPH?

GUI choice requested: Found 183 mentions of “BRAPH”. Save the full list as a Markdown file?
  * 0: Save all 183 to outputs/ as Markdown
    1: Show the first 20 here
  Decision: option 0
ASSISTANT:
Found **183 mentions** of “BRAPH” in **5 of 5 files** (ignoring case, whole
words).

**Mentions per file**

- tiunn26braph2.pdf: 85
- mijalkov17braph.pdf: 80
- mijalkov25computational.pdf: 9
- patil25deep.pdf: 7
- yao25connecting.pdf: 2

Scanned all indexed text in 5 files.

The full list of 183 mentions is saved as `outputs/mentions_braph.md`.

Resolved intent: mention_listing
Executed tool: scan_and_list
Wall time: 6.374 s

TURN: turn_0003

EXCHANGE
  question:
Which of these articles discuss BRAPH?


ANSWER
  summary:
Found **183 mentions** of “BRAPH” in **5 of 5 files** (ignoring case, whole
words).


INTENT
  intent:
mention_listing

  standalone:
False

  refers_back:
True

  related_turn_ids:
[2]


ROUTING
  shortcut:
None

  fallback:
No

In [19]:
answer = ask_question(
    session,
    "Which of these articles discuss BRAPH?",
    choice_callback=None,
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)



USER:
Which of these articles discuss BRAPH?

ASSISTANT:
<think>
Okay, the user is asking which articles discuss BRAPH. Let me look through the
provided evidence.

First, I see that there are five files mentioned: tiunn26braph2.pdf,
mijalkov17braph.pdf, mijalkov25computational.pdf, patil25deep.pdf, and
yao25connecting.pdf. The user wants to know which articles discuss BRAPH.

Looking at the evidence, the first file, mijalkov17braph.pdf, has multiple
mentions. For example, [1] and [2] talk about BRAPH as a graph theory software.
[4] and [5] also discuss BRAPH's features and methods. [8] mentions BRAPH's
strengths and clinical applications. So that's four sources from the same file.

Then, tiunn26braph2.pdf has [3] and [6] discussing BRAPH 2, its features, and
improvements over BRAPH 1. [6] specifically talks about the Genesis module and
new functionalities.

The other files: mijalkov25computational.pdf [7] doesn't mention BRAPH, it's
about reservoir computing. Similarly, yao25connectin

In [20]:
answer = ask_question(
    session,
    "Show me the source information for the BRAPH 2 article.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
Show me the source information for the BRAPH 2 article.

ASSISTANT:
<think>Model call 1:
Okay, the user is asking for the source information for the BRAPH 2 article.
Let me check the provided source texts.

Looking at the citations, there are two PDFs mentioned: [1] mijalkov17braph.pdf
and [2] tiunn26braph2.pdf. The user probably wants details about the BRAPH 2
article, which is likely in [2] since it's titled "tiunn26braph2.pdf".

In [2], the text mentions BRAPH 2's features like the Genesis module, GitHub
hosting, and collaboration aspects. The citation [22] and [23] are referenced,
which might be related to the studies mentioned. The user might need the
citation details like the author, title, and publication info. However, the
provided texts don't include the full citation details beyond the file names
and some references.

The user might be looking for the DOI, authors, or journal name. Since the
texts don't provide that, I should note that the source information isn't full

In [21]:
answer = ask_question(
    session,
    "Find passages discussing graph measures in the BRAPH articles.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Find passages discussing graph measures in the BRAPH articles.

ASSISTANT:
Found **0 mentions** of “graph measures” in **tiunn26braph2.pdf** (ignoring
case, whole words).

Scanned only tiunn26braph2.pdf, the best match for “BRAPH articles”. Name the
file to scan a different one.

Scanned all indexed text in tiunn26braph2.pdf.

Resolved intent: mention_listing
Executed tool: scan_and_list
Wall time: 6.884 s

TURN: turn_0006

EXCHANGE
  question:
Find passages discussing graph measures in the BRAPH articles.


ANSWER
  summary:
Found **0 mentions** of “graph measures” in **tiunn26braph2.pdf** (ignoring
case, whole words).


INTENT
  intent:
mention_listing

  standalone:
False

  refers_back:
False

  related_turn_ids:
[4]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

In [22]:
answer = ask_question(
    session,
    "What do those passages say about their implementation?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)


USER:
What do those passages say about their implementation?

ASSISTANT:
<think>Model call 1:
Okay, let's tackle this question. The user is asking what the passages say
about their implementation. They mentioned using the cited sentences and the
surrounding text, so I need to look at the relevant parts from the provided
source texts.

First, looking at the first source text [1], it talks about BRAPH's strengths,
like handling all aspects of graph theory analysis, from defining nodes and
edges to producing publication-ready figures and archiving results. It also
mentions two studies on patients with amnestic MCI, AD, and PD-MCI. But the
question is about implementation, so maybe the part about the GUI and pipelines
in [2] and [3] is more relevant.

In [2], there's a mention of launching the GUI for pipeline selection and the
tutorials and developer documentation. It also refers to the practical
implementation being illustrated in Section 4. The case studies in network
neuroscience are 

In [23]:
answer = ask_question(session, "Which articles discuss deep learning?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)


USER:
Which articles discuss deep learning?

GUI plan: find_passages | Read all 3710 sentences in 5 files and list the ones about “deep learning”.
  Items: 3710, batches: 44, quick items: 24
  Decision: continue
ASSISTANT:
**5 of 5 files** match “deep learning”, from 1534 matching sentences.
- **patil25deep.pdf** (630 sentences): “Deep-Learning Investigation of
Vibrational Raman Spec- tra for Plant-Stress Analysis” [90]
- **yao25connecting.pdf** (410 sentences): “In this paper, we evaluate the
evidence that genomic results for brain disorders, diseases, and traits point
at speciﬁc brain cell types.” [180]
- **mijalkov25computational.pdf** (248 sentences): “a n g i s” [41]
- **mijalkov17braph.pdf** (207 sentences): “Citation: Mijalkov M, Kakaei E,
Pereira JB, Westman E, Volpe G, for the Alzheimer’s Disease Neuroimaging
Initiative (2017) BRAPH: A graph theory software for the analysis of brain
connectivity.” [1]
- **tiunn26braph2.pdf** (39 sentences): “SoftwareX 35 (2026) 102791” [164]


In [24]:
session.close()

## Article Understanding and Synthesis

In [25]:
store, conversation, session = create_conversation(project_dir)

Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T16-50-09.json


In [26]:
answer = ask_question(session, "Give me an overview of the BRAPH 2 paper.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Give me an overview of the BRAPH 2 paper.

GUI choice requested: Which paper should I summarize?
  * 0: tiunn26braph2.pdf (best match for the question)
    1: mijalkov17braph.pdf (matches the question)
    2: mijalkov25computational.pdf (in this project)
    3: patil25deep.pdf (in this project)
  Decision: option 0
ASSISTANT:
<think>
Okay, I need to summarize the BRAPH 2 paper based on the provided passages. Let
me start by reading through the passages again to make sure I understand the
key points.

The paper introduces BRAPH 2 as a framework for network analyses in
neuroscience. It's open-source, reproducible, and community-oriented. The main
goal is to provide flexible methods for analyzing brain networks, especially as
the complexity and size of data grow. They mention deep learning, specialized
distributions, and custom pipelines as key features.

Looking at the sections, BRAPH 2 allows users to create custom pipelines and
specialized distributions using the Genesis compile

In [27]:
answer = ask_question(
    session,
    "Summarize its main contributions and limitations.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
Summarize its main contributions and limitations.

ASSISTANT:
<think>
Okay, I need to summarize the main contributions and limitations of the BRAPH 2
paper as per the provided passages. Let me go through the text again to pick
out the key points.

First, the main contributions. The paper introduces BRAPH 2 as a flexible,
open-source framework for network analysis in neuroscience. It mentions
features like multiplex graph analysis, deep learning pipelines, and the
Genesis module for creating custom pipelines and specialized distributions. The
Genesis compiler allows users to build specialized distributions with custom
pipelines, enhancing flexibility and scalability. They also highlight
reproducibility through .b2 files that encapsulate entire analyses, ensuring
exact replication across systems. The framework supports multiple data types
beyond brain imaging and is used in various domains like yeast-cell
synchronization and dendritic spine loss. The paper also discusses the impac

In [28]:
answer = ask_question(
    session,
    "Generate a detailed summary of the BRAPH 2 paper.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
Generate a detailed summary of the BRAPH 2 paper.

ASSISTANT:
Done: the technical summary of tiunn26braph2.pdf is saved as
`outputs/tiunn26braph2_summary.md`.

---

# BRAPH 2: Open-source framework for reproducible network analyses in
neuroscience

**Source paper:** Tiunn, I.-U. et al. BRAPH 2: A flexible, open-source,
reproducible, community-oriented, easy-to-use framework for network analyses in
neurosciences. (2026).

## Technical Problem and Contribution
The technical problem addressed is the need for flexible, unbiased, and
reproducible insights into brain function through network analyses in
neuroscience. Prior limitations include the lack of a framework that supports
advanced statistical models and deep learning while enabling community-oriented
extension, reuse, and sharing. The paper claims the development of BRAPH 2, a
versatile, open-source framework that overcomes these limitations by providing
a platform for standard graph analysis and custom software development. B

In [29]:
answer = ask_question(
    session,
    "How does BRAPH 2 differ from the earlier BRAPH framework?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
How does BRAPH 2 differ from the earlier BRAPH framework?

ASSISTANT:
<think>
Okay, the user is asking how BRAPH 2 differs from the earlier BRAPH framework.
Let me look through the provided evidence to find the relevant information.

First, I remember that the user previously asked about the main contributions
and limitations of BRAPH 2, and the assistant provided a detailed summary. Now,
they want a comparison between BRAPH 2 and its predecessor.

Looking at the evidence, source [8] mentions that BRAPH 2 adds multiplex graph
analysis, deep learning pipelines, and Genesis-based generation of reusable
pipelines and specialized distributions. It also highlights the Genesis module,
which compiles structured element descriptions into MATLAB classes, GUIs, and
unit tests. This seems like a key difference from the earlier version.

Source [1] talks about BRAPH 2 being implemented in MATLAB, which might be
similar to the original BRAPH, but it also mentions limitations with large
voxel

In [30]:
answer = ask_question(session, "What evidence supports those differences?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
What evidence supports those differences?

GUI plan: reference_check | Check 12 claims of tiunn26braph2.pdf against the 3 cited papers in the project, one model call each.
  Items: 12, batches: 12, quick items: 5
  Decision: continue
ASSISTANT:
<think>Model call 1:
Okay, let's tackle this. The user is asking whether the reference [19] supports
the sentence that cites it. The sentence mentions BRAPH 1 as one of the free
and open-source software platforms for brain connectivity analysis, and the
reference [19] is cited for "BRAPH 1".

First, I need to look at the provided passages from reference [19], which is
the paper titled "BRAPH: A graph theory software for the analysis of brain
connectivity" by Mijalkov et al. (2017). The key part here is that the sentence
lists BRAPH 1 as an example, and the user wants to know if the cited work
(reference [19]) supports that.

Looking at the passages, especially Passage 2, the paper lists several other
software tools like CONN [19], but wai

In [31]:
answer = ask_question(
    session,
    "Which of the four papers is most relevant to graph-based analysis "
    "of brain connectivity, and why?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Which of the four papers is most relevant to graph-based analysis of brain
connectivity, and why?

ASSISTANT:
<think>
Okay, let's tackle this question. The user is asking which of the four papers
is most relevant to graph-based analysis of brain connectivity and why. The
evidence provided includes several references, but I need to focus on the ones
that directly address graph-based analysis.

First, I'll look at the four papers mentioned. The user refers to "four
papers," but in the evidence, there are more than four. Wait, the user might
have meant the four papers listed in the evidence, but I need to check. Let me
re-read the question. The user says, "Which of the four papers is most
relevant..." but in the evidence, there are multiple references. However, the
user might have intended to refer to the four papers mentioned in the context,
which are [1] to [4], but I need to confirm.

Looking at the evidence, the user lists [1] to [20], but the question refers to
four papers. Wa

In [32]:
session.close()

## Specialized Literature Tools

In [33]:
store, conversation, session = create_conversation(project_dir)

Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T17-14-23.json


In [34]:
answer = ask_question(
    session,
    "Find the passage in the BRAPH 2 paper "
    "describing its software architecture.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Find the passage in the BRAPH 2 paper describing its software architecture.

ASSISTANT:
I couldn't find an earlier answer with sources to show. Ask the question first,
then ask to see its source.

Resolved intent: source_context
Executed tool: show_source
Wall time: 6.000 s

TURN: turn_0001

EXCHANGE
  question:
Find the passage in the BRAPH 2 paper describing its software architecture.


ANSWER
  summary:
I couldn't find an earlier answer with sources to show.


INTENT
  intent:
source_context

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
relevant

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
10


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.002


RETR

In [35]:
answer = ask_question(
    session,
    "What programming language is BRAPH 2 implemented in?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
What programming language is BRAPH 2 implemented in?

ASSISTANT:
<think>
Okay, the user is asking what programming language BRAPH 2 is implemented in.
Let me look through the provided evidence.

First, I'll scan through the evidence numbers. Let's start with [1]. It
mentions that BRAPH 2 is implemented in MATLAB. That's a direct answer. But I
should check other sources to confirm.

Looking at [3], it says the deep learning pipelines are implemented in MATLAB
within BRAPH 2. That supports the same point. [8] also states that BRAPH 2 is
written in MATLAB. [9] explicitly mentions that BRAPH 2 is written in MATLAB,
which aligns with the others.

Other sources like [2] and [4] talk about the architecture and features but
don't mention the programming language. [5] and [7] are about other studies and
tools, not BRAPH 2's implementation.

So the consistent answer across multiple sources [1], [3], [8], and [9] is
MATLAB. The user might be looking to use BRAPH 2 or understand its technic

In [36]:
answer = ask_question(session, "List the articles that mention graph theory.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
List the articles that mention graph theory.

GUI choice requested: Found 65 mentions of “graph theory”. Save the full list as a Markdown file?
    0: Save all 65 to outputs/ as Markdown
  * 1: Show the first 20 here
  Decision: option 1
ASSISTANT:
Found **65 mentions** of “graph theory” in **5 of 5 files** (ignoring case,
whole words).

**Mentions per file**

- mijalkov17braph.pdf: 59
- tiunn26braph2.pdf: 3
- mijalkov25computational.pdf: 1
- patil25deep.pdf: 1
- yao25connecting.pdf: 1

**Where**

1. mijalkov17braph.pdf, page 1, line 3: BRAPH: A **graph theory** software for
the analysis of brain connectivity [1]
2. mijalkov17braph.pdf, page 1, lines 27-28: … Alzheimer’s Disease Neuroimaging
Initiative (2017) BRAPH: A **graph theory** software for the analysis of brain
connectivity. PLoS ONE 12… [2]
3. mijalkov17braph.pdf, page 1, line 63: …ain network has been studied
extensively using concepts from **graph theory**, where the brain is
represented as a set of nodes connected… [

In [37]:
answer = ask_question(
    session,
    "Filter those results to articles discussing brain connectivity.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
Filter those results to articles discussing brain connectivity.

GUI plan: find_passages | Read all 3710 sentences in 5 files and list the ones about “brain connectivity”.
  Items: 3710, batches: 44, quick items: 20
  Decision: continue
ASSISTANT:
**5 of 5 files** match “brain connectivity”, from 1701 matching sentences.
- **patil25deep.pdf** (519 sentences): “Deep-Learning Investigation of
Vibrational Raman Spec- tra for Plant-Stress Analysis” [131]
- **yao25connecting.pdf** (430 sentences): “Connecting genomic results for
psychiatric disorders to human brain cell types and regions reveals convergence
with functional connectivity” [214]
- **mijalkov25computational.pdf** (358 sentences): “Computational memory
capacity predicts aging and cognitive decline” [67]
- **mijalkov17braph.pdf** (328 sentences): “BRAPH: A graph theory software for
the analysis of brain connectivity” [1]
- **tiunn26braph2.pdf** (66 sentences): “The standard BRAPH 2 distribution
(https://github.com/braph-so

In [38]:
answer = ask_question(
    session,
    "Classify the five articles by their main research topic.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
Classify the five articles by their main research topic.

GUI plan: find_passages | Read all 3710 sentences in 5 files and list the ones about “main research topic”.
  Items: 3710, batches: 44, quick items: 309
  Decision: continue
ASSISTANT:
**5 of 5 files** match “main research topic”, from 2496 matching sentences.
- **patil25deep.pdf** (955 sentences): “Deep-Learning Investigation of
Vibrational Raman Spec- tra for Plant-Stress Analysis” [135]
- **yao25connecting.pdf** (746 sentences): “In this paper, we evaluate the
evidence that genomic results for brain disorders, diseases, and traits point
at speciﬁc brain cell types.” [270]
- **mijalkov25computational.pdf** (357 sentences): “Computational memory
capacity predicts aging and cognitive decline” [66]
- **mijalkov17braph.pdf** (293 sentences): “Citation: Mijalkov M, Kakaei E,
Pereira JB, Westman E, Volpe G, for the Alzheimer’s Disease Neuroimaging
Initiative (2017) BRAPH: A graph theory software for the analysis of brain
conn

In [39]:
answer = ask_question(
    session,
    "Review the five articles for their relevance to a literature review "
    "on computational brain connectivity.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Review the five articles for their relevance to a literature review on
computational brain connectivity.

ASSISTANT:
<think>
Okay, let's tackle this question. The user wants me to review the five articles
for their relevance to a literature review on computational brain connectivity.
First, I need to recall the evidence provided and how each article relates to
computational brain connectivity.

Starting with the first article, [1] is from mijalkov17braph.pdf. The title is
"BRAPH: A graph theory software for the analysis of brain connectivity." The
abstract mentions graph theory software, which is directly related to
computational methods in brain connectivity. The article also discusses
functional brain networks and modules, which are key in computational
connectivity studies. So this one is definitely relevant.

Next, [2] is the same source, but it's about references and funding. The
content here is more about the methodology and references, but the main topic
is still graph th

In [40]:
answer = ask_question(
    session,
    "Check whether the BRAPH 2 paper supports the claim "
    "that it provides a graphical user interface.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)


USER:
Check whether the BRAPH 2 paper supports the claim that it provides a graphical
user interface.

GUI plan: reference_check | Check 12 claims of tiunn26braph2.pdf against the 3 cited papers in the project, one model call each.
  Items: 12, batches: 12, quick items: 5
  Decision: continue
ASSISTANT:
<think>Model call 1:
Okay, let's tackle this. The user is asking whether the reference [19] supports
the sentence that cites it. The sentence mentions BRAPH 1 as one of the free
and open-source software platforms for brain connectivity analysis, and the
reference [19] is cited for "BRAPH 1".

First, I need to look at the provided passages from reference [19], which is
the paper titled "BRAPH: A graph theory software for the analysis of brain
connectivity" by Mijalkov et al. (2017). The key part here is that the sentence
lists BRAPH 1 as an example, and the user wants to know if the cited work
(reference [19]) supports that.

Looking at the passages, especially Passage 2, the paper list

In [41]:
answer = ask_question(
    session,
    "Which article should I cite for the introduction of the BRAPH framework?",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)


USER:
Which article should I cite for the introduction of the BRAPH framework?

ASSISTANT:
<think>Model call 1:
Okay, let's tackle this. The user is asking if the candidate work
(tiunn26braph2.pdf) can be cited for the sentence: "Which article should I cite
for the introduction of the BRAPH framework?"

First, I need to look through the provided passages to see if any of them
mention citing an article or the introduction of the BRAPH framework. The
sentence is about citing a specific article for the introduction of BRAPH, so I
need to check if the passages discuss citing articles or the introduction part.

Looking at Passage 1, it talks about BRAPH 2's features, how users can download
and use it, and mentions the GUI and tutorials. There's no mention of citing an
article here. Passage 2 discusses limitations and future developments, again no
citations. Passage 3 and 4 mention the framework's features, GitHub hosting,
and the Genesis module, but again, no specific citations.

Passage 4

In [42]:
session.close()

## Continuity and Robustness

In [43]:
store, conversation, session = create_conversation(project_dir)

Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T17-53-57.json


In [44]:
answer = ask_question(session, "Summarize the BRAPH 2 article.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Summarize the BRAPH 2 article.

GUI choice requested: Which paper should I summarize?
  * 0: tiunn26braph2.pdf (best match for the question)
    1: mijalkov17braph.pdf (matches the question)
    2: mijalkov25computational.pdf (in this project)
    3: patil25deep.pdf (in this project)
  Decision: option 0
ASSISTANT:
<think>
Okay, I need to summarize the BRAPH 2 article based on the provided passages.
Let me start by reading through the passages again to make sure I understand
the key points.

The paper introduces BRAPH 2 as an open-source framework for network analyses
in neuroscience. It's designed to be flexible, reproducible, and
community-oriented. The main goal is to address the growing complexity and size
of network analyses in neuroscience by providing tools that allow unbiased and
reproducible insights into brain function.

The framework includes features like deep learning pipelines, specialized
distributions, and a Genesis compiler that helps users create custom pipelin

In [45]:
answer = ask_question(session, "What are its main limitations?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
What are its main limitations?

ASSISTANT:
<think>
Okay, the user is asking about the main limitations of BRAPH 2. Let me go
through the provided evidence to find relevant information.

First, looking at the sources, source [9] mentions that BRAPH 2 is implemented
in MATLAB, which can lead to memory demands for large voxel-level networks.
Sparse matrices help but might reduce computational efficiency. Also, Genesis,
the core of BRAPH 2, isn't suited for direct benchmarking against conventional
tools, so benchmarking is left for future work. That's one limitation.

Then, source [5] talks about the study's limitations. They used cross-sectional
data from two cohorts, which can't establish causality or predict cognitive
trajectories over time. Longitudinal studies would be better. Methodologically,
they used a homogeneous activation function for all regions, which simplifies
interpretation but might limit model complexity. They mention that
region-dependent activation functions cou

In [46]:
answer = ask_question(session, "How does it compare with the 2017 paper?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
How does it compare with the 2017 paper?

ASSISTANT:
<think>
Okay, the user is asking how BRAPH 2 compares with the 2017 paper. Let me start
by recalling what I know from the provided evidence.

First, the 2017 paper, which is referenced as [2] and [6], introduced BRAPH as
a software package for graph theory analysis in neuroscience. It emphasized a
modular structure, object-oriented programming, and features like permutation
testing, visualization, and longitudinal analyses. It also mentioned handling
multiple data types and integration with different neuroimaging modalities.

Now, BRAPH 2, as discussed in sources [1], [3], [9], [10], and [14], is an
updated version. The main differences seem to be in functionality and features.
BRAPH 2 adds multiplex graph analysis, deep learning pipelines, and the Genesis
module for creating custom pipelines and distributions. It also focuses on
reproducibility with .b2 files and supports more data types and analytical
methods. The 2017 paper

In [47]:
answer = ask_question(session, "Now tell me about the deep learning article.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
Now tell me about the deep learning article.

ASSISTANT:
<think>
Okay, the user is asking about the deep learning article. Let me look through
the provided evidence to find relevant information.

First, I see that evidence [1] and [4] mention BRAPH 2's deep learning
pipelines. They talk about supervised tasks like classification, prediction,
and diagnosis. The pipelines are implemented in MATLAB using the Deep Learning
Toolbox. There's also a mention of a multilayer perceptron used for age
prediction, which is referenced in [3] and [4].

In [3], there's a discussion about memory capacity calculation using Pearson’s
correlation coefficient, which relates to the reservoir computing approach. The
article also references variational autoencoders and other deep learning
techniques from [30] and [31], but those might be more general.

Evidence [9] and [10] discuss the application of deep learning in analyzing
memory capacity and age prediction. They mention specific models like the
mu

In [48]:
answer = ask_question(session, "What methods does it use?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
What methods does it use?

ASSISTANT:
<think>
Okay, the user is asking about the methods used in the deep learning article.
Let me look through the provided evidence to find the relevant information.

First, I remember the user mentioned the deep learning article in their
previous question, and the assistant answered by discussing the methods like
MLP, reservoir computing, and memory capacity metrics. Now, they want to know
what methods are used, so I need to focus on the evidence provided here.

Looking at the evidence, source [3] and [5] mention BRAPH 2's use of deep
learning, specialized distributions, custom pipelines, and reproducibility.
They also talk about the Genesis compiler for building workflows. Source [4]
and [6] discuss DIVA's methods, like using the first derivative of Raman
spectra to reduce fluorescence influence. However, the user is asking about the
deep learning article, so maybe the main focus is on BRAPH 2's methods.

Wait, the user's initial question was 

In [49]:
answer = ask_question(session, "Compare it with the previous article.")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Compare it with the previous article.

ASSISTANT:
<think>
Okay, the user wants me to compare the current article with the previous one.
Let me start by recalling what the previous article was about. From the
evidence provided, the previous article, which I assume is the one referenced
in [1] and [2], discussed the integration of deep learning into BRAPH 2 for
neuroimaging analysis. It mentioned specific methods like multilayer
perceptrons (MLPs) for age prediction and reservoir computing for assessing
cognitive decline using memory capacity metrics derived from Pearson’s
correlation coefficients. The focus was on supervised tasks like classification
and prediction, with an emphasis on reproducibility through the Genesis
framework and MATLAB implementations.

Now, looking at the current article, which is the one the user is asking about,
the evidence [1] and [3] mention BRAPH 2's features, including deep learning
pipelines, specialized distributions, and custom pipelines. They hi

In [50]:
answer = ask_question(
    session,
    "Find evidence that all five articles use the same dataset.",
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)


USER:
Find evidence that all five articles use the same dataset.

ASSISTANT:
Found **0 mentions** of “dataset” in **mijalkov17braph.pdf** (ignoring case,
whole words).

Scanned only mijalkov17braph.pdf, the best match for “all five articles”. Name
the file to scan a different one.

Scanned all indexed text in mijalkov17braph.pdf.

Resolved intent: mention_listing
Executed tool: scan_and_list
Wall time: 7.597 s

TURN: turn_0007

EXCHANGE
  question:
Find evidence that all five articles use the same dataset.


ANSWER
  summary:
Found **0 mentions** of “dataset” in **mijalkov17braph.pdf** (ignoring case,
whole words).


INTENT
  intent:
mention_listing

  standalone:
False

  refers_back:
True

  related_turn_ids:
[4, 5, 6]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  te

In [51]:
answer = ask_question(session, "Which article did we discuss first?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)


USER:
Which article did we discuss first?

ASSISTANT:
<think>
Okay, the user is asking which article was discussed first. Let me check the
provided evidence.

Looking at the sources, the first mention is [1] tiunn26braph2.pdf:1-34, which
is the abstract of the BRAPH 2 paper. The user's initial question was about
summarizing the BRAPH 2 article, and the first answer referenced [1] and [2].
Then, in the follow-up, they compared it with the previous article, which might
have been another source. However, the user now wants to know which article was
discussed first.

Wait, the user's first question was about summarizing the BRAPH 2 article, and
the answer used sources [1] through [19]. The follow-up question compared it
with the previous article, but the user hasn't provided a separate article. The
current context is only the BRAPH 2 article.

Wait, maybe the user is referring to the articles mentioned in the evidence.
The first source [1] is the BRAPH 2 paper, and the other sources are p

In [52]:
session.close()

## Literature Intent and Tool Coverage

This section verifies **routing and tool selection** for all 21 intents
configured in `kilin/project_types/literature.json`. The test harness
**does not force** an intent: it records the actual intent and tool saved
by `RAGSession.ask()` and prints `MATCH`, `MISMATCH`, or `ERROR`.

The GUI's **choice** and **confirmation** adapters are simulated using
explicit callbacks. The default choice shows up to 20 results in chat;
if asked to choose an ambiguous paper it selects the first offered
paper and prints that decision. The default confirmation for these
coverage tests is **Quick**, to keep batches manageable. Network lookups
are **always declined**, even if Continue is selected for model plans.

Run the groups in order so their contextual follow-ups have evidence.
The extended group (whole-paper scans, classification, reference checks,
and the four output templates) is opt-in because it can take many model
calls and write files. Set `RUN_EXTENDED_CASES = True` to exercise those
nine remaining intents. Inspect each mismatch rather than assuming a
successful answer means the intended operation ran.

A `MATCH` only validates intent and tool selection, **not** the truth
of a generated claim, citation support, report quality, or GUI rendering.
The notebook does not provide the GUI's optional Qt PDF writer; report
tools will still create their Markdown output.


In [53]:
# False: run the initial 12 intent cases; report the rest as SKIPPED.
# True: exercise every configured literature intent and its selected tool.
RUN_EXTENDED_CASES = False

# Static coverage contract: fail if the project's declared intents change.
PLANNED_INTENTS = {
    "small_talk", "question_answering", "compare_sources",
    "report_generation", "conversation_followup", "rephrase_previous",
    "project_inventory", "mention_listing", "refine_results",
    "source_context", "paper_overview", "suggest_reference",
    "topic_listing", "fact_finding", "document_reading",
    "mention_classification", "reference_check", "research_briefing",
    "popular_science_summary", "technical_summary",
    "experimental_protocol",
}
assert PLANNED_INTENTS == set(_intent_definitions()), (
    "Update the test cases when literature.json adds or removes intents."
)
print(f"Planned intent tests: {len(PLANNED_INTENTS)}")


Planned intent tests: 21


### A. Direct answers, comparison, and conversation continuity

These six tests share one conversation so the follow-up and rephrasing
intents have actual preceding answers to use. They test routing, not
whether the answer's scientific content is correct.


In [54]:
store, conversation, session = create_conversation(project_dir)

check_intent(session, "small_talk", "hello")
check_intent(
    session,
    "question_answering",
    "According to the BRAPH 2 paper, why was its software developed?",
)
check_intent(
    session,
    "compare_sources",
    "Compare the methods of mijalkov17braph.pdf and "
    "tiunn26braph2.pdf, including their differences.",
)
check_intent(
    session,
    "report_generation",
    "Write a structured report with headings about methods for "
    "brain connectivity analysis across these papers.",
)
check_intent(
    session,
    "conversation_followup",
    "In your previous report, what was the main limitation you noted?",
)
check_intent(
    session,
    "rephrase_previous",
    "Rewrite your previous answer in two concise bullet points, "
    "without adding any information.",
)

session.close()


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T18-00-22.json

USER:
hello

ASSISTANT:
Hello!

Resolved intent: small_talk
Wall time: 1.695 s
INTENT TEST: small_talk -> small_talk; tool - -> -; MATCH

USER:
According to the BRAPH 2 paper, why was its software developed?

ASSISTANT:
<think>
Okay, the user is asking why the BRAPH 2 software was developed according to
the paper. Let me look through the provided evidence to find the answer.

First, I'll scan through the evidence numbers. Evidence [9] mentions that BRAPH
2 addresses challenges in reproducibility, maintainability, and long-term
reliability. It combines a user-friendly environment with a software-generating
framework. That seems like a key point. Also, it adds features like multiplex
graph analysis and deep learning pipelines.

Looking at [1], it talks about ROI-based connectivity analyses being standard
in neuroimaging studies. BRAPH 2 is implemented in M

### B. File discovery, exact mentions, and prior-result tools

`refine_results` and `source_context` operate on saved evidence from
earlier turns. The 20-item callback matches the GUI's chat-list option.
If retrieval finds no usable mentions, inspect the corresponding tool's
result before interpreting a later failure as a routing defect.


In [55]:
store, conversation, session = create_conversation(project_dir)

check_intent(
    session,
    "project_inventory",
    "Which PDF files are available in this project?",
)
check_intent(
    session,
    "mention_listing",
    "List the sentences in tiunn26braph2.pdf that mention BRAPH.",
    choice_callback=choose_first_20,
)
check_intent(
    session,
    "source_context",
    "Show the exact source text and surrounding context for the "
    "first cited sentence in that list.",
)
check_intent(
    session,
    "refine_results",
    "Of the BRAPH mentions you listed, which ones describe "
    "software features rather than bibliographic citations?",
)

session.close()


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T18-15-43.json

USER:
Which PDF files are available in this project?

ASSISTANT:
This project has **5 source files**, all with searchable text.

**By type**

- PDF: 5

**5 PDF files**

- mijalkov17braph.pdf (78 parts, 15.8 MB)
- mijalkov25computational.pdf (93 parts, 6.3 MB)
- patil25deep.pdf (114 parts, 6.4 MB)
- tiunn26braph2.pdf (36 parts, 3.1 MB)
- yao25connecting.pdf (130 parts, 3.5 MB)

Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Resolved intent: project_inventory
Executed tool: project_inventory
Wall time: 5.714 s
INTENT TEST: project_inventory -> project_inventory; tool project_inventory -> project_inventory; MATCH

USER:
List the sentences in tiunn26braph2.pdf that mention BRAPH.

GUI choice requested: Found 85 mentions of “BRAPH”. Save the full list as a Markdown file?
    0: Save all 85 to outpu

### C. Paper overview, reference suggestion, and document selection

The third test intentionally omits the filename to exercise the GUI's
paper-selection dialog. The callback chooses `tiunn26braph2.pdf` by
its **actual option label**, not by assuming a fixed index. If the file
is not offered, it cancels instead of picking an unrelated paper.


In [56]:
store, conversation, session = create_conversation(project_dir)

check_intent(
    session,
    "paper_overview",
    "Give me a short overview of tiunn26braph2.pdf.",
)
check_intent(
    session,
    "suggest_reference",
    "Suggest a reference from this project for the statement: "
    "BRAPH supports graph analysis of brain networks.",
)
# Additional selection-path test; still expects paper_overview.
check_intent(
    session,
    "paper_overview",
    "Give me a short overview of one paper in the collection.",
    choice_callback=choose_document_named("tiunn26braph2.pdf"),
)

session.close()


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-11-01T18-16-55.json

USER:
Give me a short overview of tiunn26braph2.pdf.

ASSISTANT:
<think>
Okay, I need to summarize the paper "tiunn26braph2.pdf" based on the provided
passages. Let me start by reading through the passages to understand the main
points.

The paper introduces BRAPH 2, a framework for network analyses in neuroscience.
It's open-source, reproducible, and community-oriented. The key features
include deep learning, specialized distributions, and custom pipelines. The
main goal is to provide flexible methods for analyzing brain networks, ensuring
reproducibility and ease of use.

Looking at the sections, the paper explains how BRAPH 2 works. It uses a
Genesis compiler to create specialized distributions with custom pipelines.
Users can build their own analysis tools without extensive coding. There are
three levels of usage: standard pipelines, custom pipelines

### D. Extended tool tests (opt-in)

Enable `RUN_EXTENDED_CASES` in the setup cell to execute the nine
remaining intents. **Quick** is selected for model-intensive plans;
this verifies the quick route, not exhaustive full-processing behavior.
Reference lookup over the network is always denied. These calls may
still be slow, and report templates create Markdown files in `outputs/`.


In [57]:
extended_cases = (
    (
        "topic_listing",
        "Which papers apply graph-theory methods to analyze brain "
        "connectivity, based on their meaning rather than keywords?",
    ),
    (
        "fact_finding",
        "Give a complete list of the programming languages and software "
        "dependencies named in tiunn26braph2.pdf.",
    ),
    (
        "document_reading",
        "Read the whole tiunn26braph2.pdf paper from start to end "
        "and identify which software components it describes.",
    ),
    (
        "mention_classification",
        "Classify the mentions of BRAPH in tiunn26braph2.pdf into "
        "(1) descriptions of current software and (2) references "
        "to earlier work, then count each group.",
    ),
    (
        "reference_check",
        "Check the references cited by tiunn26braph2.pdf and determine "
        "which cited papers are present in this project.",
    ),
)

if RUN_EXTENDED_CASES:
    store, conversation, session = create_conversation(project_dir)
    try:
        for intent, question in extended_cases:
            check_intent(
                session, intent, question,
                confirm_callback=confirm_gui_quick,
            )
    finally:
        session.close()
else:
    for intent, _ in extended_cases:
        skip_intent(intent, "Set RUN_EXTENDED_CASES = True")


SKIPPED topic_listing: Set RUN_EXTENDED_CASES = True
SKIPPED fact_finding: Set RUN_EXTENDED_CASES = True
SKIPPED document_reading: Set RUN_EXTENDED_CASES = True
SKIPPED mention_classification: Set RUN_EXTENDED_CASES = True
SKIPPED reference_check: Set RUN_EXTENDED_CASES = True


### E. Four structured report tools (opt-in)

Each case targets a distinct intent but calls the same
`generate_paper_summary` tool with a different template. Markdown
reports are saved in the project workspace. Repeating the cells may
create additional output files. The notebook does not generate PDFs.


In [58]:
report_cases = (
    (
        "research_briefing",
        "Prepare a research briefing for tiunn26braph2.pdf.",
    ),
    (
        "popular_science_summary",
        "Write a popular science summary of tiunn26braph2.pdf "
        "for a general audience.",
    ),
    (
        "technical_summary",
        "Prepare a detailed technical summary of tiunn26braph2.pdf "
        "for specialists.",
    ),
    (
        "experimental_protocol",
        "Extract an experimental protocol based on tiunn26braph2.pdf; "
        "identify missing replication details without inventing them.",
    ),
)

if RUN_EXTENDED_CASES:
    store, conversation, session = create_conversation(project_dir)
    try:
        for intent, question in report_cases:
            check_intent(
                session, intent, question,
                confirm_callback=confirm_gui_quick,
            )
    finally:
        session.close()
else:
    for intent, _ in report_cases:
        skip_intent(intent, "Set RUN_EXTENDED_CASES = True")


SKIPPED research_briefing: Set RUN_EXTENDED_CASES = True
SKIPPED popular_science_summary: Set RUN_EXTENDED_CASES = True
SKIPPED technical_summary: Set RUN_EXTENDED_CASES = True
SKIPPED experimental_protocol: Set RUN_EXTENDED_CASES = True


### F. Confirmation and output-path regression (optional)

The main test groups choose **Quick** for large model plans. To test
GUI-style cancellation without an expensive scan, run the cell below:
it should produce a cancelled-tool answer without starting model batches.
The ordinary earlier BRAPH cells already exercise **show 20**, **save
all**, and the deliberately noninteractive **show up to 100** paths.


In [59]:
# Optional: the fact_finding intent always offers a plan when items exist.
# This tests a GUI Cancel choice; it is not included in matched coverage.
RUN_CANCEL_REGRESSION = False

if RUN_CANCEL_REGRESSION:
    store, conversation, session = create_conversation(project_dir)
    try:
        check_intent(
            session,
            "fact_finding",
            "List the programming languages in tiunn26braph2.pdf.",
            confirm_callback=confirm_gui_cancel,
        )
    finally:
        session.close()


### G. Coverage summary

Run this cell after the groups above. A `MATCH` means the **saved**
intent equals the intended intent and the saved tool name equals the
tool defined in `literature.json`. `INCOMPLETE` means routing matched, but the tool did not produce its expected result metadata. `SKIPPED` means the opt-in test was
not executed. If a route mismatches, inspect that question's trace and
revise the example or investigate the router; do not force the intent.


In [60]:
coverage = show_intent_coverage()

# Full selection-path coverage also requires testing Continue and Cancel
# branches on appropriate non-network tool plans. The automated suite
# above focuses on the ordinary and Quick paths.



LITERATURE INTENT COVERAGE
Intent                      Expected tool            Status
-------------------------------------------------------------------------------
question_answering          model                    MATCH
fact_finding                find_fact                SKIPPED
compare_sources             model                    MATCH
document_reading            batch_review             SKIPPED
paper_overview              paper_overview           MATCH
reference_check             reference_check          SKIPPED
suggest_reference           suggest_reference        MATCH
research_briefing           generate_paper_summary   SKIPPED
popular_science_summary     generate_paper_summary   SKIPPED
technical_summary           generate_paper_summary   SKIPPED
experimental_protocol       generate_paper_summary   SKIPPED
project_inventory           project_inventory        MATCH
mention_listing             scan_and_list            MATCH
topic_listing               find_passages          